In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import tensorflow as tf

Mounted at /content/drive


In [ ]:
RESULTS_DIR = "/content/drive/MyDrive/ASD_XAI_Project/results"

base_df = pd.read_csv(
    f"{RESULTS_DIR}/combined_xai_slice_ranking.csv"
)

slice_df = base_df[
    ["SUB_ID", "SLICE_PATH", "LABEL"]
].drop_duplicates().copy()

print("Slices:", len(slice_df))
print("Subjects:", slice_df["SUB_ID"].nunique())
print(slice_df.head())

Slices: 1054
Subjects: 6
   SUB_ID                                         SLICE_PATH    LABEL
0   51487  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control
1   51487  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control
2   51487  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control
3   51487  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control
4   51487  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

IMG_SIZE = 224

# Pretrained ResNet50 backbone
base_model = tf.keras.applications.ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Freeze pretrained layers initially
base_model.trainable = False

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

# ResNet-specific preprocessing
x = tf.keras.applications.resnet50.preprocess_input(inputs)

x = base_model(
    x,
    training=False
)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.3)(x)

outputs = layers.Dense(
    1,
    activation="sigmoid"
)(x)

resnet_model = models.Model(
    inputs,
    outputs
)

resnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("ResNet50 created!")
print("Input:", resnet_model.input_shape)
print("Output:", resnet_model.output_shape)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
ResNet50 created!
Input: (None, 224, 224, 3)
Output: (None, 1)


In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/ASD_XAI_Project"

for root, dirs, files in os.walk(PROJECT_DIR):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

/content/drive/MyDrive/ASD_XAI_Project/results/shap_validation_scores.csv
/content/drive/MyDrive/ASD_XAI_Project/results/permutation_validation_scores.csv
/content/drive/MyDrive/ASD_XAI_Project/results/combined_xai_slice_ranking.csv
/content/drive/MyDrive/ASD_XAI_Project/results/top20_xai_selected_slices.csv
/content/drive/MyDrive/ASD_XAI_Project/results/topk_validation_comparison.csv
/content/drive/MyDrive/ASD_XAI_Project/results/top5_xai_selected_slices.csv
/content/drive/MyDrive/ASD_XAI_Project/metadata/phenotypic_CALTECH.csv
/content/drive/MyDrive/ASD_XAI_Project/metadata/train_subjects.csv
/content/drive/MyDrive/ASD_XAI_Project/metadata/val_subjects.csv
/content/drive/MyDrive/ASD_XAI_Project/metadata/test_subjects.csv
/content/drive/MyDrive/ASD_XAI_Project/metadata/train_slices.csv
/content/drive/MyDrive/ASD_XAI_Project/metadata/val_slices.csv
/content/drive/MyDrive/ASD_XAI_Project/metadata/test_slices.csv
/content/drive/MyDrive/ASD_XAI_Project/experiment_02_weight_optimization/we

In [ ]:
TRAIN_CSV = "/content/drive/MyDrive/ASD_XAI_Project/metadata/train_slices.csv"
VAL_CSV = "/content/drive/MyDrive/ASD_XAI_Project/metadata/val_slices.csv"
TEST_CSV = "/content/drive/MyDrive/ASD_XAI_Project/metadata/test_slices.csv"

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

print("Train slices:", len(train_df))
print("Validation slices:", len(val_df))
print("Test slices:", len(test_df))

print("\nTrain columns:", train_df.columns.tolist())
print("\nFirst rows:")
print(train_df.head())

print("\nTrain subjects:", train_df["SUB_ID"].nunique())
print("Validation subjects:", val_df["SUB_ID"].nunique())
print("Test subjects:", test_df["SUB_ID"].nunique())

print("\nTrain label counts:")
print(train_df["LABEL"].value_counts())

Train slices: 4536
Validation slices: 1054
Test slices: 1036

Train columns: ['SUB_ID', 'SLICE_PATH', 'LABEL', 'SPLIT']

First rows:
   SUB_ID                                         SLICE_PATH    LABEL  SPLIT
0   51482  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control  train
1   51482  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control  train
2   51482  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control  train
3   51482  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control  train
4   51482  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control  train

Train subjects: 26
Validation subjects: 6
Test subjects: 6

Train label counts:
LABEL
ASD        2294
Control    2242
Name: count, dtype: int64


In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 16

def encode_label(label):
    return 1.0 if label == "ASD" else 0.0

def make_dataset(df, training=False):

    paths = df["SLICE_PATH"].values
    labels = np.array([
        encode_label(x)
        for x in df["LABEL"].values
    ], dtype=np.float32)

    ds = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    def load_and_prepare(path, label):
        image = tf.io.read_file(path)
        image = tf.image.decode_png(
            image,
            channels=3
        )
        image = tf.image.resize(
            image,
            (IMG_SIZE, IMG_SIZE)
        )
        image = tf.cast(
            image,
            tf.float32
        )

        return image, label

    ds = ds.map(
        load_and_prepare,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if training:
        ds = ds.shuffle(
            buffer_size=len(df),
            seed=42
        )

    ds = ds.batch(BATCH_SIZE)

    ds = ds.prefetch(
        tf.data.AUTOTUNE
    )

    return ds


train_ds = make_dataset(
    train_df,
    training=True
)

val_ds = make_dataset(
    val_df,
    training=False
)

print("Datasets ready!")

Datasets ready!


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models

# -----------------------------
# 1. Load metadata
# -----------------------------

TRAIN_CSV = "/content/drive/MyDrive/ASD_XAI_Project/metadata/train_slices.csv"
VAL_CSV = "/content/drive/MyDrive/ASD_XAI_Project/metadata/val_slices.csv"

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)

print("Train slices:", len(train_df))
print("Validation slices:", len(val_df))


# -----------------------------
# 2. Create ResNet50
# -----------------------------

IMG_SIZE = 224
BATCH_SIZE = 16

base_model = tf.keras.applications.ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False

inputs = layers.Input(
    shape=(IMG_SIZE, IMG_SIZE, 3)
)

x = tf.keras.applications.resnet50.preprocess_input(
    inputs
)

x = base_model(
    x,
    training=False
)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.3)(x)

outputs = layers.Dense(
    1,
    activation="sigmoid"
)(x)

resnet_model = models.Model(
    inputs,
    outputs
)

resnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("ResNet recreated!")


# -----------------------------
# 3. Create datasets
# -----------------------------

def encode_label(label):
    return 1.0 if label == "ASD" else 0.0


def make_dataset(df, training=False):

    paths = df["SLICE_PATH"].values

    labels = np.array([
        encode_label(x)
        for x in df["LABEL"].values
    ], dtype=np.float32)

    ds = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    def load_image(path, label):

        image = tf.io.read_file(path)

        image = tf.image.decode_png(
            image,
            channels=3
        )

        image = tf.image.resize(
            image,
            (IMG_SIZE, IMG_SIZE)
        )

        image = tf.cast(
            image,
            tf.float32
        )

        return image, label

    ds = ds.map(
        load_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if training:
        ds = ds.shuffle(
            len(df),
            seed=42
        )

    ds = ds.batch(BATCH_SIZE)

    ds = ds.prefetch(
        tf.data.AUTOTUNE
    )

    return ds


train_ds = make_dataset(
    train_df,
    training=True
)

val_ds = make_dataset(
    val_df,
    training=False
)


print("Datasets recreated!")

print(
    "GPU:",
    tf.config.list_physical_devices("GPU")
)

Train slices: 4536
Validation slices: 1054
ResNet recreated!
Datasets recreated!
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6
    )
]

history = resnet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks
)

In [ ]:
print("TensorFlow:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.20.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models

# -------------------------
# Load saved metadata
# -------------------------

TRAIN_CSV = "/content/drive/MyDrive/ASD_XAI_Project/metadata/train_slices.csv"
VAL_CSV = "/content/drive/MyDrive/ASD_XAI_Project/metadata/val_slices.csv"

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)

print("Train slices:", len(train_df))
print("Validation slices:", len(val_df))


# -------------------------
# ResNet50 model
# -------------------------

IMG_SIZE = 224
BATCH_SIZE = 16

base_model = tf.keras.applications.ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False

inputs = layers.Input(
    shape=(IMG_SIZE, IMG_SIZE, 3)
)

x = tf.keras.applications.resnet50.preprocess_input(inputs)

x = base_model(
    x,
    training=False
)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.3)(x)

outputs = layers.Dense(
    1,
    activation="sigmoid"
)(x)

resnet_model = models.Model(
    inputs,
    outputs
)

resnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("\nResNet50 ready!")
print(resnet_model.input_shape)


# -------------------------
# Dataset function
# -------------------------

def encode_label(label):
    return 1.0 if label == "ASD" else 0.0


def make_dataset(df, training=False):

    paths = df["SLICE_PATH"].values

    labels = np.array(
        [
            encode_label(label)
            for label in df["LABEL"].values
        ],
        dtype=np.float32
    )

    ds = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    def load_image(path, label):

        image = tf.io.read_file(path)

        image = tf.image.decode_png(
            image,
            channels=3
        )

        image = tf.image.resize(
            image,
            (IMG_SIZE, IMG_SIZE)
        )

        image = tf.cast(
            image,
            tf.float32
        )

        return image, label

    ds = ds.map(
        load_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if training:
        ds = ds.shuffle(
            len(df),
            seed=42
        )

    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds


train_ds = make_dataset(
    train_df,
    training=True
)

val_ds = make_dataset(
    val_df,
    training=False
)

print("\nDatasets ready!")

Train slices: 4536
Validation slices: 1054

ResNet50 ready!
(None, 224, 224, 3)

Datasets ready!


In [11]:
import os

print("Local slices exist:", os.path.exists("/content/slices"))

Local slices exist: False


In [12]:
!cp -r "/content/drive/MyDrive/ASD_XAI_Project/slices" "/content/slices"

In [13]:
train_df["SLICE_PATH"] = train_df["SLICE_PATH"].str.replace(
    "/content/drive/MyDrive/ASD_XAI_Project/slices",
    "/content/slices",
    regex=False
)

val_df["SLICE_PATH"] = val_df["SLICE_PATH"].str.replace(
    "/content/drive/MyDrive/ASD_XAI_Project/slices",
    "/content/slices",
    regex=False
)

print(train_df["SLICE_PATH"].iloc[0])
print(val_df["SLICE_PATH"].iloc[0])

/content/slices/Control/51482/slice_000.jpg
/content/slices/Control/51487/slice_000.jpg


In [14]:
train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)

print("Local datasets ready!")

Local datasets ready!


In [16]:
history = resnet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/10
284/284 ━━━━━━━━━━━━━━━━━━━━ 20s 57ms/step - accuracy: 0.5633 - loss: 0.6909 - val_accuracy: 0.2808 - val_loss: 0.8011 - learning_rate: 5.0000e-05
Epoch 2/10
284/284 ━━━━━━━━━━━━━━━━━━━━ 19s 57ms/step - accuracy: 0.6043 - loss: 0.6636 - val_accuracy: 0.2609 - val_loss: 0.8355 - learning_rate: 5.0000e-05
Epoch 3/10
284/284 ━━━━━━━━━━━━━━━━━━━━ 19s 59ms/step - accuracy: 0.6283 - loss: 0.6373 - val_accuracy: 0.2429 - val_loss: 0.8519 - learning_rate: 2.5000e-05


In [17]:
print("Epochs actually completed:", len(history.history["loss"]))

print("\nTraining accuracy:")
print(history.history["accuracy"])

print("\nValidation accuracy:")
print(history.history["val_accuracy"])

print("\nValidation loss:")
print(history.history["val_loss"])

Epochs actually completed: 3

Training accuracy:
[0.5632715821266174, 0.6042768955230713, 0.6283068656921387]

Validation accuracy:
[0.2808349132537842, 0.2609108090400696, 0.24288424849510193]

Validation loss:
[0.8010858297348022, 0.8354901075363159, 0.8519268035888672]


In [18]:
# Unfreeze ResNet backbone
base_model.trainable = True

# Keep most layers frozen
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Keep BatchNormalization layers frozen for stability
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False


# Recompile with MUCH smaller learning rate
resnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("Trainable layers:",
      sum(layer.trainable for layer in resnet_model.layers))

print("Fine-tuning ready!")

Trainable layers: 5
Fine-tuning ready!


In [19]:
fine_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7
    )
]

In [20]:
fine_history = resnet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    callbacks=fine_callbacks,
    verbose=1
)

Epoch 1/8
284/284 ━━━━━━━━━━━━━━━━━━━━ 58s 132ms/step - accuracy: 0.8325 - loss: 0.3411 - val_accuracy: 0.5199 - val_loss: 1.9578 - learning_rate: 1.0000e-05
Epoch 2/8
284/284 ━━━━━━━━━━━━━━━━━━━━ 27s 83ms/step - accuracy: 0.9700 - loss: 0.0798 - val_accuracy: 0.5427 - val_loss: 2.5535 - learning_rate: 1.0000e-05
Epoch 3/8
284/284 ━━━━━━━━━━━━━━━━━━━━ 28s 83ms/step - accuracy: 0.9857 - loss: 0.0336 - val_accuracy: 0.5645 - val_loss: 2.6203 - learning_rate: 1.0000e-05
Epoch 4/8
284/284 ━━━━━━━━━━━━━━━━━━━━ 28s 83ms/step - accuracy: 0.9971 - loss: 0.0084 - val_accuracy: 0.5550 - val_loss: 3.1051 - learning_rate: 5.0000e-06


In [21]:
def load_resnet_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, (224, 224))
    image = tf.cast(image, tf.float32)
    return image


resnet_subject_results = []

for sub_id, group in val_df.groupby("SUB_ID"):

    images = np.stack([
        load_resnet_image(path).numpy()
        for path in group["SLICE_PATH"]
    ])

    preds = resnet_model.predict(
        images,
        batch_size=16,
        verbose=0
    ).reshape(-1)

    subject_probability = np.mean(preds)

    predicted_label = (
        "ASD"
        if subject_probability >= 0.5
        else "Control"
    )

    true_label = group["LABEL"].iloc[0]

    resnet_subject_results.append({
        "SUB_ID": sub_id,
        "TRUE_LABEL": true_label,
        "PREDICTED_LABEL": predicted_label,
        "PROBABILITY": subject_probability,
        "NUM_SLICES": len(group)
    })


resnet_subject_df = pd.DataFrame(resnet_subject_results)

resnet_subject_accuracy = (
    resnet_subject_df["TRUE_LABEL"]
    == resnet_subject_df["PREDICTED_LABEL"]
).mean()

print("=== RESNET SUBJECT-LEVEL RESULTS ===")
print(resnet_subject_df)

print("\nSubject-level accuracy:")
print(resnet_subject_accuracy)

=== RESNET SUBJECT-LEVEL RESULTS ===
   SUB_ID TRUE_LABEL PREDICTED_LABEL  PROBABILITY  NUM_SLICES
0   51460        ASD         Control     0.044640         176
1   51463        ASD         Control     0.069810         176
2   51466        ASD         Control     0.480338         176
3   51475    Control         Control     0.231620         176
4   51476    Control         Control     0.088993         174
5   51487    Control         Control     0.267456         176

Subject-level accuracy:
0.5


In [22]:
import os

MODEL_DIR = "/content/drive/MyDrive/ASD_XAI_Project/models"
os.makedirs(MODEL_DIR, exist_ok=True)

RESNET_PATH = (
    f"{MODEL_DIR}/resnet50_finetuned.keras"
)

resnet_model.save(RESNET_PATH)

print("ResNet saved successfully!")
print("Path:", RESNET_PATH)

ResNet saved successfully!
Path: /content/drive/MyDrive/ASD_XAI_Project/models/resnet50_finetuned.keras


In [23]:
RESNET_RESULTS_DIR = (
    "/content/drive/MyDrive/ASD_XAI_Project/"
    "experiment_04_resnet50"
)

os.makedirs(
    RESNET_RESULTS_DIR,
    exist_ok=True
)

resnet_subject_df.to_csv(
    f"{RESNET_RESULTS_DIR}/resnet_baseline_subject_results.csv",
    index=False
)

print("Baseline results saved!")
print("Accuracy:", resnet_subject_accuracy)

Baseline results saved!
Accuracy: 0.5


In [24]:
!pip -q install shap

In [25]:
import shap
import tensorflow as tf
import numpy as np
import pandas as pd
import os

RESNET_PATH = (
    "/content/drive/MyDrive/ASD_XAI_Project/models/"
    "resnet50_finetuned.keras"
)

resnet_model = tf.keras.models.load_model(
    RESNET_PATH
)

VAL_CSV = (
    "/content/drive/MyDrive/ASD_XAI_Project/metadata/"
    "val_slices.csv"
)

val_slices_df = pd.read_csv(VAL_CSV)

print("ResNet loaded:", resnet_model.input_shape)
print("Validation slices:", len(val_slices_df))
print("Validation subjects:", val_slices_df["SUB_ID"].nunique())

ResNet loaded: (None, 224, 224, 3)
Validation slices: 1054
Validation subjects: 6


In [26]:
# Load training metadata for SHAP background samples
TRAIN_CSV = (
    "/content/drive/MyDrive/ASD_XAI_Project/metadata/"
    "train_slices.csv"
)

train_slices_df = pd.read_csv(TRAIN_CSV)

IMG_SIZE = 224


def load_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32)
    return image


# Use a small background set from training data
background_paths = train_slices_df["SLICE_PATH"].iloc[:20]

background_images = np.stack([
    load_image(path).numpy()
    for path in background_paths
])

print("Background shape:", background_images.shape)


# Create ResNet SHAP explainer
explainer = shap.GradientExplainer(
    resnet_model,
    background_images
)

print("SHAP explainer created!")

Background shape: (20, 224, 224, 3)
SHAP explainer created!


In [27]:
test_paths = val_slices_df["SLICE_PATH"].iloc[:2]

test_images = np.stack([
    load_image(path).numpy()
    for path in test_paths
])

test_shap = explainer.shap_values(
    test_images,
    nsamples=20
)

if isinstance(test_shap, list):
    test_shap = test_shap[0]

print("Input shape:", test_images.shape)
print("SHAP shape:", np.array(test_shap).shape)

print("ResNet SHAP test successful!")

/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input_layer_5
Received: inputs=['Tensor(shape=(2, 224, 224, 3))']
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input_layer_5
Received: inputs=['Tensor(shape=(20, 224, 224, 3))']
  warnings.warn(msg)


Input shape: (2, 224, 224, 3)
SHAP shape: (2, 224, 224, 3, 1)
ResNet SHAP test successful!


In [28]:
import os
import numpy as np
import pandas as pd

SHAP_BATCH_SIZE = 4

RESNET_SHAP_OUTPUT = (
    "/content/drive/MyDrive/ASD_XAI_Project/"
    "experiment_04_resnet50/resnet_shap_validation_scores.csv"
)

CHECKPOINT_OUTPUT = (
    "/content/drive/MyDrive/ASD_XAI_Project/"
    "experiment_04_resnet50/resnet_shap_checkpoint.csv"
)

shap_results = []

for start in range(0, len(val_slices_df), SHAP_BATCH_SIZE):

    end = min(
        start + SHAP_BATCH_SIZE,
        len(val_slices_df)
    )

    batch_df = val_slices_df.iloc[start:end]

    batch_images = np.stack([
        load_image(path).numpy()
        for path in batch_df["SLICE_PATH"]
    ])

    shap_values = explainer.shap_values(
        batch_images,
        nsamples=20
    )

    if isinstance(shap_values, list):
        shap_values = shap_values[0]

    shap_values = np.array(shap_values)

    # Remove single-output dimension
    if shap_values.ndim == 5:
        shap_values = np.squeeze(
            shap_values,
            axis=-1
        )

    # One score per MRI slice
    slice_scores = np.mean(
        np.abs(shap_values),
        axis=(1, 2, 3)
    )

    for i, (_, row) in enumerate(
        batch_df.iterrows()
    ):
        shap_results.append({
            "SUB_ID": row["SUB_ID"],
            "SLICE_PATH": row["SLICE_PATH"],
            "LABEL": row["LABEL"],
            "SHAP_SCORE": float(
                slice_scores[i]
            )
        })

    print(
        f"ResNet SHAP processed: "
        f"{end}/{len(val_slices_df)}"
    )

    # Save checkpoint
    if end % 100 < SHAP_BATCH_SIZE or end == len(val_slices_df):

        pd.DataFrame(
            shap_results
        ).to_csv(
            CHECKPOINT_OUTPUT,
            index=False
        )

        print("Checkpoint saved:", end)


resnet_shap_df = pd.DataFrame(
    shap_results
)

resnet_shap_df.to_csv(
    RESNET_SHAP_OUTPUT,
    index=False
)

print("\nResNet SHAP completed!")
print("Rows:", len(resnet_shap_df))

print(
    resnet_shap_df
    .sort_values(
        "SHAP_SCORE",
        ascending=False
    )
    .head(10)
)

/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input_layer_5
Received: inputs=['Tensor(shape=(4, 224, 224, 3))']
  warnings.warn(msg)


ResNet SHAP processed: 4/1054
ResNet SHAP processed: 8/1054
ResNet SHAP processed: 12/1054
ResNet SHAP processed: 16/1054
ResNet SHAP processed: 20/1054
ResNet SHAP processed: 24/1054
ResNet SHAP processed: 28/1054
ResNet SHAP processed: 32/1054
ResNet SHAP processed: 36/1054
ResNet SHAP processed: 40/1054
ResNet SHAP processed: 44/1054
ResNet SHAP processed: 48/1054
ResNet SHAP processed: 52/1054
ResNet SHAP processed: 56/1054
ResNet SHAP processed: 60/1054
ResNet SHAP processed: 64/1054
ResNet SHAP processed: 68/1054
ResNet SHAP processed: 72/1054
ResNet SHAP processed: 76/1054
ResNet SHAP processed: 80/1054
ResNet SHAP processed: 84/1054
ResNet SHAP processed: 88/1054
ResNet SHAP processed: 92/1054
ResNet SHAP processed: 96/1054
ResNet SHAP processed: 100/1054
Checkpoint saved: 100
ResNet SHAP processed: 104/1054
ResNet SHAP processed: 108/1054
ResNet SHAP processed: 112/1054
ResNet SHAP processed: 116/1054
ResNet SHAP processed: 120/1054
ResNet SHAP processed: 124/1054
ResNet SHAP 

In [29]:
import numpy as np
import pandas as pd

PERM_BATCH_SIZE = 16

RESNET_PERM_OUTPUT = (
    "/content/drive/MyDrive/ASD_XAI_Project/"
    "experiment_04_resnet50/resnet_permutation_validation_scores.csv"
)

permutation_results = []

rng = np.random.default_rng(42)

for start in range(0, len(val_slices_df), PERM_BATCH_SIZE):

    end = min(
        start + PERM_BATCH_SIZE,
        len(val_slices_df)
    )

    batch_df = val_slices_df.iloc[start:end]

    batch_images = np.stack([
        load_image(path).numpy()
        for path in batch_df["SLICE_PATH"]
    ])

    # Original predictions
    original_preds = resnet_model.predict(
        batch_images,
        batch_size=PERM_BATCH_SIZE,
        verbose=0
    ).reshape(-1)

    # Permute each image
    permuted_images = batch_images.copy()

    for i in range(len(permuted_images)):

        flat = permuted_images[i].reshape(-1).copy()

        rng.shuffle(flat)

        permuted_images[i] = flat.reshape(
            permuted_images[i].shape
        )

    # Predictions after permutation
    permuted_preds = resnet_model.predict(
        permuted_images,
        batch_size=PERM_BATCH_SIZE,
        verbose=0
    ).reshape(-1)

    permutation_scores = np.abs(
        original_preds - permuted_preds
    )

    for i, (_, row) in enumerate(batch_df.iterrows()):

        permutation_results.append({
            "SUB_ID": row["SUB_ID"],
            "SLICE_PATH": row["SLICE_PATH"],
            "LABEL": row["LABEL"],
            "PERMUTATION_SCORE": float(
                permutation_scores[i]
            )
        })

    print(
        f"ResNet permutation processed: "
        f"{end}/{len(val_slices_df)}"
    )


resnet_perm_df = pd.DataFrame(
    permutation_results
)

resnet_perm_df.to_csv(
    RESNET_PERM_OUTPUT,
    index=False
)

print("\nResNet permutation completed!")
print("Rows:", len(resnet_perm_df))

print("\nTop permutation-ranked slices:")

print(
    resnet_perm_df
    .sort_values(
        "PERMUTATION_SCORE",
        ascending=False
    )
    .head(10)
)

ResNet permutation processed: 16/1054
ResNet permutation processed: 32/1054
ResNet permutation processed: 48/1054
ResNet permutation processed: 64/1054
ResNet permutation processed: 80/1054
ResNet permutation processed: 96/1054
ResNet permutation processed: 112/1054
ResNet permutation processed: 128/1054
ResNet permutation processed: 144/1054
ResNet permutation processed: 160/1054
ResNet permutation processed: 176/1054
ResNet permutation processed: 192/1054
ResNet permutation processed: 208/1054
ResNet permutation processed: 224/1054
ResNet permutation processed: 240/1054
ResNet permutation processed: 256/1054
ResNet permutation processed: 272/1054
ResNet permutation processed: 288/1054
ResNet permutation processed: 304/1054
ResNet permutation processed: 320/1054
ResNet permutation processed: 336/1054
ResNet permutation processed: 352/1054
ResNet permutation processed: 368/1054
ResNet permutation processed: 384/1054
ResNet permutation processed: 400/1054
ResNet permutation processed: 4

In [30]:
# Merge ResNet SHAP + Permutation

resnet_combined_df = resnet_shap_df.merge(
    resnet_perm_df[
        ["SUB_ID", "SLICE_PATH", "PERMUTATION_SCORE"]
    ],
    on=["SUB_ID", "SLICE_PATH"],
    how="inner"
)

# Normalize SHAP
shap_min = resnet_combined_df["SHAP_SCORE"].min()
shap_max = resnet_combined_df["SHAP_SCORE"].max()

resnet_combined_df["SHAP_NORM"] = (
    (resnet_combined_df["SHAP_SCORE"] - shap_min)
    / (shap_max - shap_min)
)

# Normalize permutation
perm_min = resnet_combined_df["PERMUTATION_SCORE"].min()
perm_max = resnet_combined_df["PERMUTATION_SCORE"].max()

resnet_combined_df["PERMUTATION_NORM"] = (
    (resnet_combined_df["PERMUTATION_SCORE"] - perm_min)
    / (perm_max - perm_min)
)

print("Rows:", len(resnet_combined_df))

print(
    "SHAP normalized range:",
    resnet_combined_df["SHAP_NORM"].min(),
    resnet_combined_df["SHAP_NORM"].max()
)

print(
    "Permutation normalized range:",
    resnet_combined_df["PERMUTATION_NORM"].min(),
    resnet_combined_df["PERMUTATION_NORM"].max()
)

Rows: 1054
SHAP normalized range: 0.0 1.0
Permutation normalized range: 0.0 1.0


In [31]:
WEIGHT_VALUES = [
    (0.2, 0.8),
    (0.3, 0.7),
    (0.4, 0.6),
    (0.5, 0.5),
    (0.6, 0.4),
    (0.7, 0.3),
    (0.8, 0.2)
]

K_VALUES = [5, 10, 15, 20]

resnet_xai_results = []

for alpha, beta in WEIGHT_VALUES:

    temp_df = resnet_combined_df.copy()

    temp_df["XAI_SCORE"] = (
        alpha * temp_df["SHAP_NORM"]
        + beta * temp_df["PERMUTATION_NORM"]
    )

    temp_df["XAI_RANK"] = (
        temp_df
        .groupby("SUB_ID")["XAI_SCORE"]
        .rank(method="first", ascending=False)
    )

    for k in K_VALUES:

        selected = temp_df[
            temp_df["XAI_RANK"] <= k
        ].copy()

        results = []

        for sub_id, group in selected.groupby("SUB_ID"):

            images = np.stack([
                load_image(path).numpy()
                for path in group["SLICE_PATH"]
            ])

            preds = resnet_model.predict(
                images,
                batch_size=16,
                verbose=0
            ).reshape(-1)

            probability = np.mean(preds)

            predicted = (
                "ASD"
                if probability >= 0.5
                else "Control"
            )

            results.append({
                "TRUE": group["LABEL"].iloc[0],
                "PRED": predicted
            })

        results_df = pd.DataFrame(results)

        accuracy = (
            results_df["TRUE"] == results_df["PRED"]
        ).mean()

        resnet_xai_results.append({
            "SHAP_WEIGHT": alpha,
            "PERM_WEIGHT": beta,
            "TOP_K": k,
            "ACCURACY": accuracy
        })

        print(
            f"SHAP={alpha:.1f} | "
            f"PERM={beta:.1f} | "
            f"K={k:2d} | "
            f"Accuracy={accuracy:.4f}"
        )

resnet_xai_results_df = pd.DataFrame(
    resnet_xai_results
)

best = resnet_xai_results_df.loc[
    resnet_xai_results_df["ACCURACY"].idxmax()
]

print("\n=== BEST RESNET + XAI RESULT ===")
print(best)

SHAP=0.2 | PERM=0.8 | K= 5 | Accuracy=0.5000
SHAP=0.2 | PERM=0.8 | K=10 | Accuracy=0.5000
SHAP=0.2 | PERM=0.8 | K=15 | Accuracy=0.5000
SHAP=0.2 | PERM=0.8 | K=20 | Accuracy=0.5000
SHAP=0.3 | PERM=0.7 | K= 5 | Accuracy=0.5000
SHAP=0.3 | PERM=0.7 | K=10 | Accuracy=0.5000
SHAP=0.3 | PERM=0.7 | K=15 | Accuracy=0.5000
SHAP=0.3 | PERM=0.7 | K=20 | Accuracy=0.5000
SHAP=0.4 | PERM=0.6 | K= 5 | Accuracy=0.5000
SHAP=0.4 | PERM=0.6 | K=10 | Accuracy=0.5000
SHAP=0.4 | PERM=0.6 | K=15 | Accuracy=0.5000
SHAP=0.4 | PERM=0.6 | K=20 | Accuracy=0.5000
SHAP=0.5 | PERM=0.5 | K= 5 | Accuracy=0.5000
SHAP=0.5 | PERM=0.5 | K=10 | Accuracy=0.5000
SHAP=0.5 | PERM=0.5 | K=15 | Accuracy=0.5000
SHAP=0.5 | PERM=0.5 | K=20 | Accuracy=0.5000
SHAP=0.6 | PERM=0.4 | K= 5 | Accuracy=0.5000
SHAP=0.6 | PERM=0.4 | K=10 | Accuracy=0.5000
SHAP=0.6 | PERM=0.4 | K=15 | Accuracy=0.5000
SHAP=0.6 | PERM=0.4 | K=20 | Accuracy=0.5000
SHAP=0.7 | PERM=0.3 | K= 5 | Accuracy=0.5000
SHAP=0.7 | PERM=0.3 | K=10 | Accuracy=0.5000
SHAP=0.7 |

In [32]:
RESNET_XAI_OUTPUT = (
    "/content/drive/MyDrive/ASD_XAI_Project/"
    "experiment_04_resnet50/resnet_xai_weight_results.csv"
)

resnet_xai_results_df.to_csv(
    RESNET_XAI_OUTPUT,
    index=False
)

print("ResNet XAI results saved!")

ResNet XAI results saved!
